# Module 13 — Layer Normalization

Stack enough layers on top of each other (and Module 17's nanoGPT will
stack several transformer blocks) and a real problem shows up: activation
values can drift to very different scales from one layer to the next,
making training unstable or painfully slow. **Layer normalization**
re-centers and re-scales every token's feature vector — independently, per
token — to have mean 0 and variance 1, then applies a learned scale
(`gamma`) and shift (`beta`) so the network can undo the normalization if
that turns out to be useful.

The key word is *per token*: layer norm computes its statistics across the
**embedding dimension** of a single token, not across a batch of examples
(that's *batch* norm, a different technique). This is exactly why it works
for language models — it doesn't care about batch size, and sequences can
be any length.

## 1. The problem: unnormalized activations drift in scale

To see the effect clearly, use a weight initialization that's a bit too
large for this depth (a very realistic mistake — the "right" init scale
depends on depth and width, and it's easy to get wrong) and stack several
plain linear layers with no nonlinearity in between, so nothing else is
masking the effect.

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(42)

d_model = 16
n_layers = 8
x = torch.randn(1, d_model)

layers = [nn.Linear(d_model, d_model, bias=False) for _ in range(n_layers)]
for layer in layers:
    nn.init.normal_(layer.weight, mean=0.0, std=2.2 / (d_model ** 0.5))

print("Activation std at each layer, WITHOUT normalization:")
h = x
for i, layer in enumerate(layers):
    h = layer(h)
    print(f"  layer {i}: std = {h.std().item():.2f}")

## 2. Layer normalization, implemented from scratch

For a feature vector `v` of length `d_model`: subtract its own mean,
divide by its own standard deviation, then apply a learned per-dimension
scale and shift. `eps` avoids dividing by zero if variance is tiny.

In [ ]:
class LayerNormFromScratch(nn.Module):
    def __init__(self, d_model, eps=1e-5):
        super().__init__()
        self.gamma = nn.Parameter(torch.ones(d_model))
        self.beta = nn.Parameter(torch.zeros(d_model))
        self.eps = eps

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        x_norm = (x - mean) / torch.sqrt(var + self.eps)
        return self.gamma * x_norm + self.beta


ln = LayerNormFromScratch(d_model)
sample = torch.randn(4, d_model) * 10 + 5  # deliberately large mean/scale
normed = ln(sample)

per_token_mean = normed.mean(dim=-1)
per_token_std = normed.std(dim=-1, unbiased=False)
print("per-token mean after norm (should be ~0):", per_token_mean)
print("per-token std after norm (should be ~1):", per_token_std)
assert torch.allclose(per_token_mean, torch.zeros(4), atol=1e-5)
assert torch.allclose(per_token_std, torch.ones(4), atol=1e-4)
print("\nConfirmed: every token independently has mean 0, std 1 across its own feature dimension.")

## 3. Verifying against `torch.nn.LayerNorm`

The from-scratch version should match PyTorch's built-in exactly, once the
same `gamma`/`beta` are used.

In [ ]:
torch_ln = nn.LayerNorm(d_model)
torch_ln.weight.data.copy_(ln.gamma.data)
torch_ln.bias.data.copy_(ln.beta.data)

out_scratch = ln(sample)
out_torch = torch_ln(sample)
assert torch.allclose(out_scratch, out_torch, atol=1e-5)
print("From-scratch LayerNorm matches nn.LayerNorm exactly.")

## 4. Layer norm keeps the same deep stack well-behaved

Same layers, same (too-large) weight init from step 1 — the only change is
a `LayerNorm` after each linear.

In [ ]:
print("Activation std at each layer, WITH layer norm after each linear:")
norm_layers = [nn.LayerNorm(d_model) for _ in range(n_layers)]
h = x
for i, (layer, norm) in enumerate(zip(layers, norm_layers)):
    h = norm(layer(h))
    print(f"  layer {i}: std = {h.std().item():.2f}")

print("\nCompare to step 1 - instead of compounding into the hundreds, the scale is pinned to ~1 at every layer, regardless of how poorly the weights were initialized.")

## Recap

- Layer norm normalizes each token's own feature vector to mean 0,
  variance 1, then rescales with learned `gamma`/`beta` — verified to
  match `nn.LayerNorm` exactly.
- Unlike batch norm, it operates per-token, independent of batch size —
  essential for variable-length sequences in language models.
- It keeps activation scale consistent as you stack more layers, which
  Module 16's transformer block (and Module 17's stack of them) relies on
  to actually be trainable.

Next up — Module 14: residual connections, the other piece that makes deep
stacks of transformer blocks trainable at all.